In [1]:
import os
import pandas as pd
import numpy as np

PLATFORM = "kaggle"  # set to "local" when running locally
if PLATFORM == "local":
    DATA_ROOT = "./input/hms-harmful-brain-activity-classification"
else:
    DATA_ROOT = "/kaggle/input/hms-harmful-brain-activity-classification"

TRAIN_PATH = os.path.join(DATA_ROOT, "train.csv")
TEST_PATH = os.path.join(DATA_ROOT, "test.csv")
SUBMISSION_PATH = "submission.csv"




In [2]:
df = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

TARGETS = df.columns[
    -6:
]  # seizure_vote, lpd_vote, gpd_vote, lrda_vote, grda_vote, other_vote
print("Targets:", list(TARGETS))

train = (
    df.groupby("eeg_id")
    .agg(
        {
            "spectrogram_id": "first",
            "spectrogram_label_offset_seconds": "min",
            "eeg_label_offset_seconds": "median",
            "patient_id": "first",
            **{t: "sum" for t in TARGETS},
        }
    )
    .reset_index()
)

vote_vals = train[TARGETS].values
vote_vals = vote_vals / vote_vals.sum(axis=1, keepdims=True)
train[TARGETS] = vote_vals

global_avg = train[TARGETS].mean().values  # shape (6,)

patient_sum = train.groupby("patient_id")[TARGETS].sum()
patient_means = patient_sum.div(patient_sum.sum(axis=1), axis=0)  # normalize rows
patient_counts = train.groupby("patient_id").size()  # number of eeg_id rows per patient




Targets: ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']


In [3]:
test = test.rename(columns={"spectrogram_id": "spec_id"})
test_preds = (
    test[["patient_id"]]
    .merge(patient_means, left_on="patient_id", right_index=True, how="left")
    .merge(
        patient_counts.rename("cnt"), left_on="patient_id", right_index=True, how="left"
    )
)

patient_pred_vals = test_preds[TARGETS].values
nan_mask = np.isnan(patient_pred_vals)
if nan_mask.any():
    patient_pred_vals[nan_mask] = np.take(global_avg, np.where(nan_mask)[1])

# Increase patient‑specific weight by lowering alpha
alpha = 0.5
counts = test_preds["cnt"].fillna(0).values
blend_weight = counts / (counts + alpha)
blend_weight = blend_weight[:, np.newaxis]  # shape (n,1)

preds = blend_weight * patient_pred_vals + (1 - blend_weight) * global_avg

epsilon = 1e-6
preds = np.clip(preds, epsilon, None)

# Remove extra smoothing: set temperature to 1.0 (no scaling)
temperature = 1.0
preds = np.power(preds, 1.0 / temperature)
preds = preds / preds.sum(axis=1, keepdims=True)




In [4]:
sub = pd.DataFrame({"eeg_id": test["eeg_id"].values})
sub[TARGETS] = preds
sub.to_csv(SUBMISSION_PATH, index=False)

print("Submission written to:", SUBMISSION_PATH)
print("Submission shape:", sub.shape)
print("Row sums (first 5 rows, should be ~1.0):")
print(sub[TARGETS].sum(axis=1).head())

Submission written to: submission.csv
Submission shape: (9850, 7)
Row sums (first 5 rows, should be ~1.0):
0    1.0
1    1.0
2    1.0
3    1.0
4    1.0
dtype: float64
